# Xarray-Spatial Multispectral: Mahalanobis distance

Anomaly detection in multispectral imagery often fails when you look at bands one at a time, because the anomalous pixels have values that overlap with the background. Mahalanobis distance solves this by measuring how unusual a pixel is across all bands jointly, accounting for correlations between them. This notebook walks through the intuition with scatter plots, then applies it to raster data.

### What you'll build

1. See why Euclidean distance fails on correlated data
2. Watch Mahalanobis distance fix the problem with covariance-aware ellipses
3. Generate a synthetic 3-band raster with a hidden anomaly patch
4. Detect the anomaly with `xrspatial.mahalanobis`
5. Compare single-band z-scores against multivariate distance
6. Use a train-apply workflow with reference statistics from a known region

![Mahalanobis distance preview](images/mahalanobis_preview.png)

[The problem with Euclidean distance](#The-problem-with-Euclidean-distance) · [Mahalanobis fixes this](#Mahalanobis-fixes-this) · [Synthetic multi-band raster](#Synthetic-multi-band-raster) · [Apply Mahalanobis distance](#Apply-Mahalanobis-distance) · [Z-scores vs. Mahalanobis](#Z-scores-vs.-Mahalanobis) · [Train-apply workflow](#Train-apply-workflow) · [Accessor syntax](#Accessor-syntax)

Standard imports plus `Ellipse` for the scatter-plot diagrams.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Patch, Rectangle

import xrspatial
from xrspatial import mahalanobis

np.random.seed(42)

## The problem with Euclidean distance

Consider two spectral bands (NIR and Red) for a patch of healthy vegetation. The reflectance values are correlated: when NIR goes up, Red tends to go up a bit too. If we draw Euclidean iso-distance circles around the mean, a point along the correlation axis gets the same score as a point perpendicular to it. One is a normal vegetation pixel and the other is anomalous, but Euclidean distance can't tell them apart.

The plot below puts two test points at the same Euclidean distance from the mean.

In [ ]:
# Generate correlated 2D data (healthy vegetation: NIR vs Red)
n_points = 500
mean_2d = np.array([0.45, 0.15])  # NIR high, Red low
cov_2d = np.array([[0.010, 0.006],
                    [0.006, 0.005]])
data_2d = np.random.multivariate_normal(mean_2d, cov_2d, n_points)

# Two test points at the same Euclidean distance from the mean
# but very different in terms of the population
eigvals, eigvecs = np.linalg.eigh(cov_2d)
r = 0.16
point_along = mean_2d + r * eigvecs[:, 1]       # along major axis (normal)
point_perp = mean_2d + r * eigvecs[:, 0]         # along minor axis (unusual)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.3, s=10, c='steelblue',
           label='Healthy vegetation')

# Euclidean iso-distance circles
for radius in [0.08, 0.12, 0.16]:
    circle = plt.Circle(mean_2d, radius, fill=False, color='grey',
                        linestyle='--', linewidth=1)
    ax.add_patch(circle)

ax.plot(*point_along, 'o', ms=12, color='darkorange', zorder=5,
        label=f'Point A (along correlation), Euclid = {r:.2f}')
ax.plot(*point_perp, 's', ms=12, color='crimson', zorder=5,
        label=f'Point B (perpendicular), Euclid = {r:.2f}')

ax.set_xlabel('NIR reflectance')
ax.set_ylabel('Red reflectance')
ax.set_title('Euclidean distance treats both points the same')
ax.legend(loc='upper left', fontsize=9)
ax.set_aspect('equal')
ax.set_xlim(0.2, 0.7)
ax.set_ylim(-0.05, 0.35)
plt.tight_layout()

Both points sit on the same grey circle: Euclidean distance says they're equally far from the mean. But Point B (red square) is clearly outside the cloud of normal data.

## Mahalanobis fixes this

[Mahalanobis distance](https://en.wikipedia.org/wiki/Mahalanobis_distance) uses the covariance of the data to stretch the distance metric. Instead of circles, we get ellipses that follow the shape of the data cloud. Now the two points get different scores.

The plot below colors each scatter point by its Mahalanobis distance and draws 1, 2, and 3 sigma ellipses.

In [ ]:
# Compute Mahalanobis distance for every point in the scatter
inv_cov_2d = np.linalg.inv(cov_2d)

def mahal_2d(pts, mu, inv_c):
    diff = pts - mu
    return np.sqrt(np.sum(diff @ inv_c * diff, axis=1))

md_data = mahal_2d(data_2d, mean_2d, inv_cov_2d)
md_A = mahal_2d(point_along.reshape(1, -1), mean_2d, inv_cov_2d)[0]
md_B = mahal_2d(point_perp.reshape(1, -1), mean_2d, inv_cov_2d)[0]

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(data_2d[:, 0], data_2d[:, 1], c=md_data, cmap='viridis',
                alpha=0.5, s=10)
plt.colorbar(sc, ax=ax, label='Mahalanobis distance')

# Mahalanobis iso-distance ellipses
for n_std in [1, 2, 3]:
    # Ellipse width is rotated by angle, so it must match the major eigenvalue
    w, h = 2 * n_std * np.sqrt(eigvals[::-1])
    angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
    ell = Ellipse(mean_2d, w, h, angle=angle, fill=False,
                  color='grey', linestyle='--', linewidth=1)
    ax.add_patch(ell)

ax.plot(*point_along, 'o', ms=12, color='darkorange', zorder=5,
        label=f'Point A (along correlation), Mahal = {md_A:.1f}')
ax.plot(*point_perp, 's', ms=12, color='crimson', zorder=5,
        label=f'Point B (perpendicular), Mahal = {md_B:.1f}')

ax.set_xlabel('NIR reflectance')
ax.set_ylabel('Red reflectance')
ax.set_title('Mahalanobis distance separates normal from anomalous')
ax.legend(loc='upper left', fontsize=9)
ax.set_aspect('equal')
ax.set_xlim(0.2, 0.7)
ax.set_ylim(-0.05, 0.35)
plt.tight_layout()

Point A sits inside the 2-sigma ellipse (low Mahalanobis distance) while Point B lands far outside (high Mahalanobis distance). Same Euclidean distance, very different Mahalanobis distances.

## Synthetic multi-band raster

Now we move from scatter plots to raster data. We'll build a synthetic 200x200 pixel image with 3 correlated bands (simulating NIR, Red, and Green reflectance), then embed an anomaly patch where the band relationships differ. The anomaly has lower NIR and higher Red, mimicking bare soil in a vegetation background.

The three individual bands are plotted below.

In [ ]:
np.random.seed(42)

H, W = 200, 200

# Background: correlated bands (healthy vegetation)
mu_bg = np.array([0.45, 0.15, 0.10])  # NIR, Red, Green
cov_bg = np.array([[0.010, 0.004, 0.003],
                    [0.004, 0.005, 0.002],
                    [0.003, 0.002, 0.004]])

pixels_bg = np.random.multivariate_normal(mu_bg, cov_bg, H * W)
pixels_bg = np.clip(pixels_bg, 0, 1)

# Reshape to (3, H, W)
bands_3d = pixels_bg.T.reshape(3, H, W)

# Anomaly patch (60x40 rectangle): different correlation structure
# (e.g. bare soil: lower NIR, higher Red)
mu_anom = np.array([0.25, 0.28, 0.22])
cov_anom = np.array([[0.004, -0.002, 0.001],
                      [-0.002, 0.006, 0.001],
                      [0.001, 0.001, 0.003]])
patch_h, patch_w = 60, 40
r0, c0 = 100, 120
n_patch = patch_h * patch_w
pixels_anom = np.random.multivariate_normal(mu_anom, cov_anom, n_patch)
pixels_anom = np.clip(pixels_anom, 0, 1)
bands_3d[:, r0:r0+patch_h, c0:c0+patch_w] = pixels_anom.T.reshape(3, patch_h, patch_w)

# Create xarray DataArrays (one per band)
band_names = ['NIR', 'Red', 'Green']
bands = []
for i, name in enumerate(band_names):
    da = xr.DataArray(bands_3d[i], dims=['y', 'x'], name=name)
    bands.append(da)

# Visualize the 3 bands
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (b, name) in enumerate(zip(bands, band_names)):
    b.plot.imshow(ax=axes[i], cmap='gray', vmin=0, vmax=0.6,
                  add_colorbar=True, cbar_kwargs={'shrink': 0.7})
    axes[i].set_title(f'Band {i+1}: {name}')
    axes[i].set_axis_off()
plt.tight_layout()

The anomaly patch (lower-right area) is hard to spot in individual bands because its values overlap with the background range. The difference is in the *correlation structure* between bands, not in individual band values.

## Apply Mahalanobis distance

`xrspatial.mahalanobis` computes the per-pixel Mahalanobis distance from a list of bands. When called without explicit `mean` or `inv_cov`, it computes statistics from all valid pixels in the input bands.

The plot shows the distance map with a `magma` colormap. Brighter pixels are further from the population mean.

In [ ]:
result = mahalanobis(bands)

fig, ax = plt.subplots(figsize=(10, 7.5))
result.plot.imshow(ax=ax, cmap='magma', add_colorbar=True,
                   cbar_kwargs={'label': 'Mahalanobis distance', 'shrink': 0.7})
ax.set_title('Mahalanobis distance (stats from all pixels)')
ax.set_axis_off()
plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/mahalanobis_preview.png',
            bbox_inches='tight', dpi=120)

The anomaly patch lights up clearly. No single band showed an obvious difference, but the multivariate distance picks it up because the *relationship* between bands is different there.

## Z-scores vs. Mahalanobis

Per-band z-scores are the standard univariate way to flag unusual values. They normalize each band independently, ignoring correlations. The comparison below shows why that's not enough: the anomaly blends into the background in each individual z-score map, while Mahalanobis distance (right panel) isolates it cleanly.

In [ ]:
# Per-band z-scores vs. Mahalanobis
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, (b, name) in enumerate(zip(bands, band_names)):
    z = (b - float(b.mean())) / float(b.std())
    z.plot.imshow(ax=axes[i], cmap='RdBu_r', vmin=-3, vmax=3,
                  add_colorbar=True, cbar_kwargs={'shrink': 0.7})
    axes[i].set_title(f'{name} z-score')
    axes[i].set_axis_off()

result.plot.imshow(ax=axes[3], cmap='magma', add_colorbar=True,
                   cbar_kwargs={'shrink': 0.7})
axes[3].set_title('Mahalanobis distance')
axes[3].set_axis_off()

plt.tight_layout()

The z-score maps show the anomaly patch blending with the background. No single band cleanly isolates it. Mahalanobis distance (right panel) catches it because it considers all bands jointly.

<div class="alert alert-block alert-warning">
<b>Singular covariance.</b> If two bands are perfectly correlated (or one has zero variance), the covariance matrix is singular and can't be inverted. <code>mahalanobis</code> will raise a <code>ValueError</code> in that case. Drop the redundant band or add a small amount of noise before computing.
</div>

## Train-apply workflow

In practice you often want to compute statistics from a **training region** (a known land-cover type, for example) and then score the entire image against that reference. Pixels similar to the training class score low; dissimilar ones score high.

Here we use the top-left quadrant (all background, no anomaly) as the training region and pass its `mean` and `inv_cov` to `mahalanobis`.

In [ ]:
# Extract training pixels from the top-left quadrant
train_region = np.stack([b.values[:100, :100] for b in bands])  # (3, 100, 100)
train_flat = train_region.reshape(3, -1).T  # (10000, 3)

# Compute reference statistics
mu = train_flat.mean(axis=0)
cov = np.cov(train_flat, rowvar=False)
inv_cov = np.linalg.inv(cov)

print('Training mean:', np.round(mu, 4))
print('Training covariance:\n', np.round(cov, 5))

In [ ]:
# Apply training stats to the full raster
result_trained = mahalanobis(bands, mean=mu, inv_cov=inv_cov)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

result.plot.imshow(ax=axes[0], cmap='magma', add_colorbar=True,
                   cbar_kwargs={'label': 'Mahalanobis distance', 'shrink': 0.7})
axes[0].set_title('Auto stats (all pixels)')
axes[0].set_axis_off()

result_trained.plot.imshow(ax=axes[1], cmap='magma', add_colorbar=True,
                           cbar_kwargs={'label': 'Mahalanobis distance', 'shrink': 0.7})
axes[1].set_title('Training stats (top-left quadrant)')
axes[1].set_axis_off()

# Outline the training region
rect = Rectangle((0, 0), 100, 100, linewidth=2, edgecolor='cyan',
                  facecolor='none', linestyle='--')
axes[1].add_patch(rect)
axes[1].text(5, 10, 'training\nregion', color='cyan', fontsize=9, va='top')

plt.tight_layout()

The train-apply result (right) uses statistics from only the training region (dashed cyan box), so the anomaly patch stands out even more. This workflow is useful when you have a known reference class and want to find pixels that deviate from it.

<div class="alert alert-block alert-info">
<b>Training region size.</b> The inverse covariance estimate needs enough pixels to be stable. As a rule of thumb, you want at least 10 times as many training pixels as bands. With only a handful of samples the covariance matrix will be noisy, and Mahalanobis distances become unreliable.
</div>

## Accessor syntax

As an alternative to the function call, you can use the `.xrs` accessor. The first band is the calling DataArray; the remaining bands go in a list. The accessor also accepts `mean` and `inv_cov` keyword arguments.

In [ ]:
# Equivalent to: mahalanobis([bands[0], bands[1], bands[2]])
result_acc = bands[0].xrs.mahalanobis([bands[1], bands[2]])

# Verify they are identical
print('Results match:', np.allclose(result.values, result_acc.values))

With explicit training stats:
```python
bands[0].xrs.mahalanobis([bands[1], bands[2]], mean=mu, inv_cov=inv_cov)
```

### References

- [Mahalanobis distance](https://en.wikipedia.org/wiki/Mahalanobis_distance), Wikipedia
- [Anomaly detection in hyperspectral imagery](https://ieeexplore.ieee.org/document/1044706), Reed & Yu, IEEE Signal Processing Magazine
- [xrspatial.mahalanobis API docs](https://makepath.github.io/xarray-spatial/reference/_autosummary/xrspatial.mahalanobis.html)